# Descriptors and regressors

How you *describe* a molecule to a model can matter more than which model you
pick!

---
### Setup

In [ ]:
#@title Getting things all setup...
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn mordredcommunity
!git clone https://github.com/agura-alt/ai4chem_openadmet.git
%cd ai4chem_openadmet


In [ ]:
#@title Imports...
import os, sys
SETUP_DIR = os.path.abspath("Setup")
os.path.isdir(SETUP_DIR) or sys.exit(f"No Setup dir at {SETUP_DIR}; cwd is {os.getcwd()}")

if SETUP_DIR not in sys.path:
    sys.path.insert(0, SETUP_DIR)

assert os.path.exists("Setup/common.py") and os.path.getsize("Setup/common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)            # force a fresh read
import common
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style("whitegrid")

Change `your-pair-name` to your team name. It has to match the list of registered teams exactly, and be the same in every notebook &mdash; that is what links your work together.

In [ ]:
# Same folder as notebooks 00 and 01 -- your splits, predictions and scores.
common.setup(pair="your-pair-name")

---
## 1. Three ways to describe a molecule

**RDKit descriptors** (~210 numbers) &mdash; interpretable physical quantities:
molecular weight, cLogP, topological polar surface area, rotatable bonds,
ring counts. These encode exactly the chemistry that drives ADMET, which is
why a simple model on these is such a strong baseline.

**Morgan fingerprints** (2048 bits) &mdash; a bit is set if a particular
substructure is present. Captures *what groups are there*, not bulk
properties.

**Mordred** (~1600 descriptors) &mdash; a much bigger physicochemical set. Slow to
compute, so it is precomputed for you.

There are many more descriptors you could use, but here's just a sampling!

In [ ]:
# load train and test!
train = common.load_train()
test = common.load_test()

### RDKit descriptors

Roughly 210 numbers per molecule, each one a physical quantity a chemist would
recognise: molecular weight, cLogP, topological polar surface area, counts of
rotatable bonds and rings. Fast to compute, and interpretable &mdash; if the
model leans on one, you can go and look at what it means.

Molecules that fail to parse come back as a row of `NaN` rather than being
dropped, so row `i` still lines up with molecule `i`.


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def rdkit_desc(smiles) -> pd.DataFrame:
    """~200 fast RDKit physicochemical descriptors."""
    names = np.array(Descriptors.descList)[:,0] # just to label our columns!

    calc = Descriptors.CalcMolDescriptors

    rows = []
    for smi in smiles: # iterate over each smiles
        mol = Chem.MolFromSmiles(smi) # convert to mol object
        empty_desc = {n: np.nan for n in names}
        if mol is None:
            rows.append(empty_desc)
        else:
            desc = calc(mol) # calculate descriptors
            rows.append(desc) # add to output

    out = pd.DataFrame(rows, columns=names).astype(float)
    return out.replace([np.inf, -np.inf], np.nan)

X_rdkit_train = rdkit_desc(train["SMILES"])
X_rdkit_test = rdkit_desc(test["SMILES"])

print("rdkit :", X_rdkit_train.shape)
X_rdkit_train.head()

### Morgan fingerprints

2048 bits. Each bit says "this particular substructure is present", found by
walking outward from every atom to a given `radius`. This captures *what groups
are there* rather than bulk properties, so it is strong on exactly the
endpoints RDKit descriptors are weak on.

Both knobs are worth turning later: `radius` sets how large a substructure a
bit can represent, `n_bits` how many distinct ones fit before different
substructures start colliding into the same bit. If you have limited data, it is worth considering whether shrinking the dimensionality by reducing `n_bits` gives you a better model!


In [ ]:
from rdkit.Chem import rdFingerprintGenerator
def morgan_desc(smiles, radius=2, n_bits=2048):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = []
    ### TODO ###
    for ...:
        mol = ...
    ### END TODO ###
        empty_desc = None
        if mol is None:
            fps.append(empty_desc)
        else:
            fp = gen.GetFingerprint(mol)# calculate descriptors
            fps.append(fp) # add to output
    arr = common.fingerprints_to_array(fps) # convert to numeric
    return pd.DataFrame(arr, columns=[f"bit_{i}" for i in range(arr.shape[1])])

X_morgan_train = morgan_desc(train["SMILES"])
X_morgan_test = morgan_desc(test["SMILES"])

print("morgan:", X_morgan_train.shape)
X_morgan_train.head()

### Mordred

About 1600 descriptors &mdash; a much larger physicochemical set than RDKit's,
overlapping with it but reaching much further into topology, connectivity and
graph-theoretic territory.

Run it on a handful of molecules first, so you can see what it actually
produces and how long it takes per molecule. Then we will load the precomputed
version for all 7608.

In [ ]:
# Mordred on a dozen molecules, so you can see the shape of the thing.
from mordred import Calculator, descriptors
import time

sample = train.head(12)

calc = Calculator(descriptors, ignore_3D=True)   # 2D only; 3D needs conformers
mols = [... for smi in sample["SMILES"]]

start = time.time()
mordred_sample = calc.pandas(mols, quiet=True)
elapsed = time.time() - start

print(f"{mordred_sample.shape[1]} descriptors for {len(mols)} molecules "
      f"in {elapsed:.1f}s  ({elapsed / len(mols):.2f}s each)")

mordred_sample.iloc[:, :12]        # first few columns

**Many columns are constant or empty** across a small sample. Over the full
dataset 240 of the 1613 columns turn out to be all-NaN or single-valued and get
dropped, which is part of why the saved file has 1373 and not 1613.

Now the precomputed version &mdash; same calculation, already done for every
molecule in train and test:

In [ ]:
MORDRED_PATH = common.data_path(os.path.join("artifacts", "mordred_descriptors.parquet"))
mordred = pd.read_parquet(MORDRED_PATH)
mordred.head()

In [ ]:
# one file covers train and test; line each up with its own frame
X_mordred_train = mordred.loc[mordred["Molecule Name"].isin(train["Molecule Name"])]
X_mordred_test = mordred.loc[mordred["Molecule Name"].isin(test["Molecule Name"])]

# drop the Molecule Name column, which is not a descriptor
X_mordred_train = X_mordred_train.drop(columns="Molecule Name")
X_mordred_test = X_mordred_test.drop(columns="Molecule Name")

print("mordred:", X_mordred_train.shape)

**Go further:** Consider other descriptors!
- [COSMO-RS](https://pubs.acs.org/jpclcd/article-abstract/11/21/9408/625031/COSMO-RS-Based-Descriptors-for-the-Machine?redirectedFrom=fulltext)
- [Jazzy](https://jazzy.readthedocs.io/en/latest/cookbook.html)
- ... and many more!

---
## 2. Regressors

Everything below is scored on **one split, held fixed**, so the only thing
changing between rows is the representation or the model. Pick it here.

These are the splits you saved in `01_validation` &mdash; if the table is empty
you have not run that notebook yet, and `"random"` is the only name that will
work.

In [ ]:
# Which splits you have depends on what you saved in 01_validation -- and on
# using the SAME pair name there as here, since they live in that pair's folder.
print("your folder:", common.workdir())
display(common.list_splits())

if common.list_splits().empty:
    print("\nNo saved splits. Only the built-in 'random' will load.\n"
          "Run 01_validation first (same pair name), or carry on with random.")

In [ ]:
SPLIT = "random"        # <-- change to a custom split, or default to random
fold, split_meta = common.load_split(train, name=SPLIT)

Let's build a model for all endpoints!
Steps will be:
1. Select your molecular representation
2. Split the rows into train and validation
3. Clean the input data
4. Initialize your model, fit, and predict.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from lightgbm import LGBMRegressor

# specify which representation you want to use!
X_all = X_rdkit_train

# split the rows into train and validation
is_train = (fold == "train").to_numpy()
is_val   = (fold == "val").to_numpy()

# clean input data -- here, the default cleaning
X_train, X_val = common.clean_features(X_all[is_train], X_all[is_val])

preds = pd.DataFrame({"Molecule Name": train.loc[is_val, "Molecule Name"].to_numpy()})
for endpoint in common.ENDPOINTS:
    y_train = train.loc[is_train, endpoint]
    ok = y_train.notna().to_numpy()          # only molecules with this measurement

    X_train_filtered = X_train[ok]                          # filter the train set
    y_filtered = y_train[ok]                          # filter the train labels

    ### initialize - fit - predict ###
    model = LGBMRegressor(n_estimators=400, learning_rate=0.05,
                            num_leaves=31, verbose=-1, n_jobs=-1)
    model.fit(X_train_filtered, y_filtered)
    preds[endpoint] = model.predict(X_val)
    ### end ###

truth = train.loc[is_val].reset_index(drop=True)
eval = common.score(truth, preds, "lgbm-rdkit", SPLIT)
print(f"MA-RAE = {eval['RAE'].mean():.3f}")
eval.round(3)

### The same loop as a function

You have now written that loop for one type of regressor. Wouldn't it be nice to swap out for a new kind?

As long as your regressor is a default scikit-learn regressor, you can write a function that handles the model training for you.

```python
preds, truth = run_model(X_rdkit_tr, LGBMRegressor(...), fold)
```

Swap the first argument to change the representation, the second to change the
model, the third to change the split!


In [ ]:
from sklearn.base import clone

def run_model(X, model, fold):
    """Fit `model` on the train rows, predict the val rows. One model per endpoint.

    X     : a descriptor frame, one row per training molecule
    model : any scikit-learn-style regressor -- it gets cloned, so the same
            one can be passed in again without carrying its old fit
    fold  : a "train"/"val" Series from common.load_split

    Returns (predictions, truth), lined up on molecule.
    """
    is_train = (fold == "train").to_numpy()
    is_val   = (fold == "val").to_numpy()
    X_train, X_val = common.clean_features(X[is_train], X[is_val])

    preds = pd.DataFrame({"Molecule Name": train.loc[is_val, "Molecule Name"].to_numpy()})
    for endpoint in common.ENDPOINTS:
        y_train = train.loc[is_train, endpoint]
        ok = y_train.notna().to_numpy()
        model = clone(model)

        X_train_filtered = X_train[ok]                          # filter the train set
        y_filtered = y_train[ok]

        model.fit(X_train_filtered, y_filtered)
        preds[endpoint] = model.predict(X_val)

    return preds, train.loc[is_val].reset_index(drop=True)

In [ ]:
# using the function with the same lgbm model as above!

# initializing a model
lgbm = LGBMRegressor(n_estimators=400, learning_rate=0.05,
                  num_leaves=31, verbose=-1, n_jobs=-1)


preds, truth = run_model(X_rdkit_train, lgbm, fold)
common.score(truth, preds, "lgbm-rdkit", SPLIT).round(3)

### &#9654;&#65039; Swap in your own regressor

LightGBM is one choice out of dozens. Here's a non-exhaustive list of other regressors built-in with the scikit-learn package (so the syntax and usage will be similar):
[scikit-learn regressor list](https://scikit-learn.org/stable/supervised_learning.html)

- `RandomForestRegressor`, `RidgeCV`,
`KNeighborsRegressor`, `GradientBoostingRegressor`, `SVR` are all reasonable
starting points. Import them, and you can fill in the rest!

- Everything else stays the same. Any scikit-learn regressor has `.fit(X, y)` and
`.predict(X)`, which is the only thing this cell needs from it.

- If you want to try a different type of regressor, feel free to not use the previous function and write it from scratch yourself!


In [ ]:
# import ...

# initializing a model
model = ...

# specify training set
preds, truth = run_model(..., model, fold)

common.score(truth, preds, "CHANGE-ME", SPLIT).round(3)


### &#9654;&#65039; Predict first

**For LogD specifically: will RDKit descriptors or Morgan fingerprints win? For Log_Caco_ER (efflux)?**

*LogD is additive and driven by bulk properties. Efflux depends on whether P-gp recognises a specific 3D arrangement. Which representation suits which?*

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

This cell may take a few minutes!

In [ ]:
REPRESENTATIONS = {
    "rdkit":   X_rdkit_train, # these names are what gets saved in common.score
    "morgan":  X_morgan_train,
    "mordred": X_mordred_train,
}

# initializing a model -- feel free to pick a different one!
model = LGBMRegressor(n_estimators=400, learning_rate=0.05,
                      num_leaves=31, verbose=-1, n_jobs=-1)

per_endpoint = {}
for name, X in REPRESENTATIONS.items():
    preds, truth = run_model(X, model, fold)
    metrics = common.score(truth, preds, f"lgbm-{name}", SPLIT,
                           note=f"LightGBM on {name}")
    per_endpoint[name] = metrics["RAE"]

pd.DataFrame(per_endpoint).round(3)      # per endpoint, per representation


Read that table **per endpoint**, not just by the average. Different
endpoints will prefer different representations!

Now vary the model with the representation held fixed.

In [ ]:
# Now hold the representation fixed and vary the model instead.
MODELS = {
    "lgbm":  LGBMRegressor(n_estimators=400, learning_rate=0.05,
                           num_leaves=31, verbose=-1, n_jobs=-1),
    "FILL_ME": ...,
    "FILL_ME_2": ...
}

for model_name, model in MODELS.items():
    preds, truth = run_model(X_rdkit_train, model, fold)
    common.score(truth, preds, f"{model_name}-rdkit", SPLIT,
                         note=f"{model_name} on rdkit descriptors")

common.score_matrix().round(3)           # everything you have logged today


Compare the spread you just saw from changing the *model* with the
spread from changing the *features*.

### Combining descriptor sets

Concatenating RDKit descriptors with fingerprints often beats either alone:
the descriptors carry bulk physics, the bits carry substructure.

Feel free to try other combinations!

In [ ]:
# Concatenate along the columns -- same molecules, more numbers each.
X_combo_train = pd.concat([X_rdkit_train, X_morgan_train], axis=1)
X_combo_test = pd.concat([X_rdkit_test, X_morgan_test], axis=1)
print("rdkit+morgan:", X_combo_train.shape)

preds, truth = run_model(X_combo_train, lgbm, fold)
common.score(truth, preds, "lgbm-rdkit+morgan", SPLIT,
                     note="LightGBM on rdkit + morgan")
common.score_matrix().round(3)


---
## Score this model against several splits

One number from one split is thin evidence. Before you decide what this
notebook was worth, run the same model through a few of your saved splits and
log all of them &mdash; that is the row that shows up in `common.score_matrix()`
next to everything else you have built today.

This cell also takes a few minutes!

In [ ]:
SPLITS = [s for s in ["random", "temporal", "similarity"]
          if s in common.known_split_names()]      # skip ones you never saved

X_best_train = ...
BEST_NAME = ...

for split_name in SPLITS:                          # note: not SPLIT -- that stays
    fold, _ = common.load_split(train, name=split_name, verbose=False)
    preds, truth = run_model(X_best_train, lgbm, fold)
    metrics = common.score(truth, preds, BEST_NAME, split_name)
    print(f"  {split_name:12s} MA-RAE = {metrics['RAE'].mean():.3f}")

In [ ]:
common.score_matrix().round(3)


## Now, prepare a submission

In [ ]:
# fit on ALL labelled data -- the splits above were only for estimating
X_best_train = ...
X_best_test = ...
SPLIT = ...                  # the split whose score you are betting on
BEST_NAME = ...

X_train, X_test = common.clean_features(X_best_train, X_best_test)

# a submission is just the molecule ids plus one column per endpoint
test_preds = pd.DataFrame({"Molecule Name": test["Molecule Name"]})

for endpoint in common.ENDPOINTS:
    y = train[endpoint]
    ok = y.notna().to_numpy()
    model = clone(lgbm)        # a fresh, unfitted copy for each endpoint
    model.fit(X_train[ok], y[ok])
    test_preds[endpoint] = model.predict(X_test)

common.prepare_submission(test_preds, BEST_NAME, SPLIT,
                          why="best representation/model from Descriptors")

test_preds.head()

If you like your submission, drag it over to the Submissions to Score folder and see how you did!